# Smoke Test — End-to-End Validation
10 FEVER examples, real API calls, no plots. Goal: verify all modules are wired correctly before running full experiments.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

N_SMOKE           = 10    # intentionally small for fast smoke validation
SMOKE_POISON_RATE = 0.5   # fixed mid-range rate to verify poisoning wiring
MODEL             = cfg["models"]["default"]
PROMPT_TYPE       = cfg["prompts"]["default"]
TEMPERATURE       = cfg["models"]["temperature"]

print("Config loaded:", list(cfg.keys()))
print(f"model={MODEL}  prompt={PROMPT_TYPE}  n_smoke={N_SMOKE}  poison_rate={SMOKE_POISON_RATE}")

Config loaded: ['seed', 'dataset', 'retrieval', 'poisoning', 'models', 'prompts', 'evaluation', 'cache', 'inference']
model=Qwen/Qwen2.5-1.5B-Instruct  prompt=standard  n_smoke=10  poison_rate=0.5


In [2]:
from src.data.fever_loader import load_fever

examples = load_fever(
    "../" + cfg["dataset"]["fever_dev"],
    max_examples=N_SMOKE,
)

print(f"Loaded {len(examples)} examples")
print("Keys:", list(examples[0].keys()))
print("Labels:", [e["label"] for e in examples])

Loaded 10 examples
Keys: ['claim', 'evidence', 'label']
Labels: ['REFUTES', 'REFUTES', 'SUPPORTS', 'NOT ENOUGH INFO', 'NOT ENOUGH INFO', 'SUPPORTS', 'NOT ENOUGH INFO', 'REFUTES', 'SUPPORTS', 'NOT ENOUGH INFO']


In [3]:
from src.data.poisoner import poison_dataset

poisoned = poison_dataset(examples, poison_rate=SMOKE_POISON_RATE, seed=cfg["seed"])
print(f"Poisoned {len(poisoned)} examples at rate {SMOKE_POISON_RATE}")

Poisoned 10 examples at rate 0.5


In [4]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever

emb_cache = os.path.join("../" + cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])

print(f"Embedder: {cfg["retrieval"]["embedding_model"]}, k={cfg["retrieval"]["k"]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedder: all-MiniLM-L6-v2, k=5


In [ ]:
from src.generation.llm_client import HuggingFaceClient

llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
llm = HuggingFaceClient(model=MODEL, temperature=TEMPERATURE, cache_dir=llm_cache)
print(f"LLM: {MODEL}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM: Qwen/Qwen2.5-1.5B-Instruct


: 

In [ ]:
from src.evaluation.scorer import run as run_scorer

with embedder, llm:
    metrics = run_scorer(
        examples=examples,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        distractor_pool_size=cfg["retrieval"]["distractor_pool_size"],
        seed=cfg["seed"],
        self_consistency_runs=1,
    )

print("\n=== Smoke Test Results (poison_rate=0.0) ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
# Quick sanity check: same run with poison_rate=SMOKE_POISON_RATE
with embedder, llm:
    metrics_poisoned = run_scorer(
        examples=poisoned,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        distractor_pool_size=cfg["retrieval"]["distractor_pool_size"],
        seed=cfg["seed"],
        self_consistency_runs=1,
    )

print(f"\n=== Smoke Test Results (poison_rate={SMOKE_POISON_RATE}) ===")
for k, v in metrics_poisoned.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n✓ Pipeline wired correctly — proceed to notebook 01." if metrics_poisoned else "✗ Check errors above.")


=== Smoke Test Results (poison_rate=0.5) ===
  accuracy: 0.5000
  macro_f1: 0.3718
  hallucination_rate: 0.0000
  precision_at_k: 0.1400

✓ Pipeline wired correctly — proceed to notebook 01.
